# Leaky integrate-and-fire notebook

This notebook mirrors `lif_model.py` and explores how the membrane potential and firing rate depend on the input current.

Requirements: numpy, matplotlib.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class LIFNeuron:
    def __init__(self, v_rest=-65.0, v_reset=-65.0, v_th=-50.0, r=10.0, tau=10.0, dt=0.1):
        self.v_rest = v_rest
        self.v_reset = v_reset
        self.v_th = v_th
        self.r = r
        self.tau = tau
        self.dt = dt
        self.v = v_rest

    def reset(self):
        self.v = self.v_reset

    def step(self, i_ext):
        dv = (-(self.v - self.v_rest) + self.r * i_ext) / self.tau
        self.v += dv * self.dt
        spike = False
        if self.v >= self.v_th:
            spike = True
            self.v = self.v_reset
        return self.v, spike

def simulate(neuron, i_ext, t_ms=500):
    n_steps = int(t_ms / neuron.dt)
    v_trace = np.zeros(n_steps)
    spikes = np.zeros(n_steps, dtype=bool)
    neuron.reset()
    for t in range(n_steps):
        v_trace[t], spikes[t] = neuron.step(i_ext)
    return v_trace, spikes


In [ ]:
neuron = LIFNeuron(dt=0.1)

i_ext = 2.0
v, spikes = simulate(neuron, i_ext)

t = np.arange(v.size) * neuron.dt

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, v)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Membrane potential (mV)')
ax.set_title(f'LIF voltage trace (I_ext={i_ext})')
plt.show()

fig, ax = plt.subplots(figsize=(8, 1.5))
spike_times = t[spikes]
ax.vlines(spike_times, 0, 1)
ax.set_yticks([])
ax.set_xlabel('Time (ms)')
ax.set_title('Spike raster')
plt.show()


In [ ]:
currents = np.linspace(0, 5, 20)
rates = []
for i in currents:
    _, spikes = simulate(neuron, i, t_ms=1000)
    rates.append(spikes.sum() / (spikes.size * neuron.dt / 1000.0))

plt.figure(figsize=(5, 3))
plt.plot(currents, rates, marker='o')
plt.xlabel('Input current (I_ext)')
plt.ylabel('Firing rate (Hz)')
plt.title('f-I curve')
plt.show()
